
# 01 — FakeMusicCaps Data Exploration

**Goal:** Explore the FakeMusicCaps dataset, examine its metadata and audio characteristics, and identify potential data quality issues before feature extraction.

**Dataset:** FakeMusicCaps — AI-generated music from five text-to-music generators.

This notebook covers:
- Dataset structure and generator distribution
- Metadata fields and prompt relationships
- Missing values and duplicate checks
- Technical audio characteristics
- Preprocessing decisions


In [2]:
%pip install soundfile

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 12.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf


In [ ]:
DATA_DIR = Path("../data/raw/fakemusiccaps")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")

In [6]:
generator_dirs = sorted(folder for folder in DATA_DIR.iterdir() if folder.is_dir() and folder.name != "__MACOSX")
generator_counts = pd.DataFrame([{"generator": folder.name, "num_files": len(list(folder.glob("*.wav")))} for folder in generator_dirs])
display(generator_counts)
print("Total files:", generator_counts["num_files"].sum())

,generator,num_files
0,audioldm2,5521
1,MusicGen_medium,5521
2,musicldm,5521
3,mustango,5521
4,stable_audio_open,5521


Total files: 27605


The FakeMusicCaps dataset contains 27,605 audio files generated by five text-to-music models: AudioLDM2, MusicGen Medium, MusicLDM, Mustango, and Stable Audio Open. Each generator contributes 5,521 audio files, resulting in a balanced dataset in terms of generator classes.

The extracted archive also contains a `__MACOSX` directory with auxiliary files, which was excluded from the analysis.

In [7]:
records = []
for folder in generator_dirs:
    for file_path in sorted(folder.glob("*.wav")):
        records.append({
            "track_id": file_path.stem,
            "generator": folder.name,
            "path": str(file_path),
        })

audio_df = pd.DataFrame(records)
display(audio_df.head())
print("Dataset shape:", audio_df.shape)

,track_id,generator,path
0,-0Gj8-vB1q4,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0Gj8-vB1q...
1,-0SdAVK79lg,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0SdAVK79l...
2,-0vPFx-wRRI,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0vPFx-wRR...
3,-0xzrMun0Rs,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0xzrMun0R...
4,-1LrH01Ei1w,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-1LrH01Ei1...


Dataset shape: (27605, 3)


### Track ID Consistency

The dataset contains 5,521 unique track IDs, each represented across all five generators. This confirms that all generators share the same set of track identifiers.

This structure enables comparisons between generators using corresponding music examples. It should also be considered when creating train, validation, and test splits to avoid potential data leakage.

In [ ]:
track_coverage = (audio_df.groupby("track_id")["generator"].nunique())
display(track_coverage.value_counts().sort_index())

generator
5    5521
Name: count, dtype: int64

### Missing Values and Duplicate Checks

In [9]:
print("Missing values:")
display(audio_df.isna().sum())

print("Duplicate generator-track pairs:")
print(audio_df.duplicated(subset=["generator", "track_id"]).sum())

Missing values:


track_id     0
generator    0
path         0
dtype: int64

Duplicate generator-track pairs:
0


### Metadata Exploration

This section examines the available metadata and the relationship between audio files, generators, and text captions.

In [10]:
metadata_files = sorted(path for path in DATA_DIR.rglob("*") if path.is_file()
    and path.suffix.lower() in {".csv", ".json", ".jsonl", ".parquet"}
    and "__MACOSX" not in path.parts
)
print("Metadata files found:", len(metadata_files))

for path in metadata_files:
    print(path.relative_to(DATA_DIR))

Metadata files found: 0


In [ ]:
METADATA_PATH = Path("../data/raw/musiccaps-public.csv")
METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)

url = ("https://huggingface.co/datasets/google/MusicCaps/resolve/main/musiccaps-public.csv")

if not METADATA_PATH.exists():
    metadata_df = pd.read_csv(url)
    metadata_df.to_csv(METADATA_PATH, index=False)
else:
    metadata_df = pd.read_csv(METADATA_PATH)

print("Metadata shape:", metadata_df.shape)
display(metadata_df.head())

Metadata shape: (5521, 9)


,ytid,start_s,end_s,audioset_positive_labels,aspect_list,caption,author_id,is_balanced_subset,is_audioset_eval
0,-0Gj8-vB1q4,30,40,"/m/0140xf,/m/02cjck,/m/04rlf","['low quality', 'sustained strings melody', 's...",The low quality recording features a ballad so...,4,False,True
1,-0SdAVK79lg,30,40,"/m/0155w,/m/01lyv,/m/0342h,/m/042v_gx,/m/04rlf...","['guitar song', 'piano backing', 'simple percu...",This song features an electric guitar as the m...,0,False,False
2,-0vPFx-wRRI,30,40,"/m/025_jnm,/m/04rlf","['amateur recording', 'finger snipping', 'male...",a male voice is singing a melody with changing...,6,False,True
3,-0xzrMun0Rs,30,40,"/m/01g90h,/m/04rlf","['backing track', 'jazzy', 'digital drums', 'p...",This song contains digital drums playing a sim...,6,False,True
4,-1LrH01Ei1w,30,40,"/m/02p0sh1,/m/04rlf","['rubab instrument', 'repetitive melody on dif...",This song features a rubber instrument being p...,0,False,False


In [12]:
print("Columns:")
print(metadata_df.columns.tolist())
print("\nMissing values:")
display(metadata_df.isna().sum())
print("\nDuplicate IDs:")
print(metadata_df["ytid"].duplicated().sum())
print("\nUnique IDs:")
print(metadata_df["ytid"].nunique())

Columns:
['ytid', 'start_s', 'end_s', 'audioset_positive_labels', 'aspect_list', 'caption', 'author_id', 'is_balanced_subset', 'is_audioset_eval']

Missing values:


ytid                        0
start_s                     0
end_s                       0
audioset_positive_labels    0
aspect_list                 0
caption                     0
author_id                   0
is_balanced_subset          0
is_audioset_eval            0
dtype: int64


Duplicate IDs:
0

Unique IDs:
5521


In [13]:
audio_ids = set(audio_df["track_id"])
metadata_ids = set(metadata_df["ytid"])

matched_ids = audio_ids & metadata_ids

print("FakeMusicCaps IDs:", len(audio_ids))
print("MusicCaps IDs:", len(metadata_ids))
print("Matching IDs:", len(matched_ids))
print("Audio IDs without metadata:", len(audio_ids - metadata_ids))
print("Metadata IDs without audio:", len(metadata_ids - audio_ids))

FakeMusicCaps IDs: 5521
MusicCaps IDs: 5521
Matching IDs: 5521
Audio IDs without metadata: 0
Metadata IDs without audio: 0


In [ ]:
audio_metadata_df = audio_df.merge(metadata_df[["ytid", "caption"]], left_on="track_id", right_on="ytid", how="left", validate="many_to_one",).drop(columns="ytid")
display(audio_metadata_df.head())
print("Shape:", audio_metadata_df.shape)
print("Missing captions:", audio_metadata_df["caption"].isna().sum())

,track_id,generator,path,caption
0,-0Gj8-vB1q4,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0Gj8-vB1q...,The low quality recording features a ballad so...
1,-0SdAVK79lg,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0SdAVK79l...,This song features an electric guitar as the m...
2,-0vPFx-wRRI,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0vPFx-wRR...,a male voice is singing a melody with changing...
3,-0xzrMun0Rs,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-0xzrMun0R...,This song contains digital drums playing a sim...
4,-1LrH01Ei1w,audioldm2,..\data\raw\fakemusiccaps\audioldm2\-1LrH01Ei1...,This song features a rubber instrument being p...


Shape: (27605, 4)
Missing captions: 0


### Metadata Exploration — Summary

All 5,521 track IDs were successfully matched between FakeMusicCaps and MusicCaps. The resulting dataset contains 27,605 records with four columns: `track_id`, `generator`, `path`, and `caption`.

No missing captions, duplicate `(generator, track_id)` pairs, or repeated captions across different track IDs were found. This confirms that the metadata is consistent and can be used for further analysis.

In [10]:
from pathlib import Path

DATA_DIR = Path("../data/raw/fakemusiccaps")

generator_dirs = [p for p in DATA_DIR.iterdir() if p.is_dir()]

for folder in generator_dirs:
    wav_files = list(folder.glob("*.wav"))
    print(f"{folder.name}: {len(wav_files)} files")

stable_audio_open: 5521 files
musicldm: 5521 files
mustango: 5521 files
audioldm2: 5521 files
MusicGen_medium: 5521 files


Each generator folder contains 5521 audio files, so the dataset is balanced across the five generator classes.

In [11]:
import soundfile as sf

for folder in generator_dirs:
    wav_file = next(folder.glob("*.wav"))
    audio, sr = sf.read(wav_file)

    duration = len(audio) / sr
    channels = 1 if audio.ndim == 1 else audio.shape[1]

    print(f"\nGenerator: {folder.name}")
    print(f"File: {wav_file.name}")
    print(f"Shape: {audio.shape}")
    print(f"Sample rate: {sr} Hz")
    print(f"Duration: {duration:.2f} s")
    print(f"Channels: {channels}")
    print(f"Dtype: {audio.dtype}")


Generator: stable_audio_open
File: dHGAXJ9RPJs.wav
Shape: (160000,)
Sample rate: 16000 Hz
Duration: 10.00 s
Channels: 1
Dtype: float64

Generator: musicldm
File: dHGAXJ9RPJs.wav
Shape: (160000,)
Sample rate: 16000 Hz
Duration: 10.00 s
Channels: 1
Dtype: float64

Generator: mustango
File: dHGAXJ9RPJs.wav
Shape: (163872,)
Sample rate: 16000 Hz
Duration: 10.24 s
Channels: 1
Dtype: float64

Generator: audioldm2
File: dHGAXJ9RPJs.wav
Shape: (160000,)
Sample rate: 16000 Hz
Duration: 10.00 s
Channels: 1
Dtype: float64

Generator: MusicGen_medium
File: dHGAXJ9RPJs.wav
Shape: (162880,)
Sample rate: 16000 Hz
Duration: 10.18 s
Channels: 1
Dtype: float64


All checked audio files are mono and use a sample rate of 16 kHz. The duration is very similar across generators, although MusicGen Medium and Mustango produce slightly longer clips than the other models.

In [12]:
filename = "dHGAXJ9RPJs.wav"

for folder in DATA_DIR.iterdir():
    if folder.is_dir():
        file_path = folder / filename

        if file_path.exists():
            print(folder.name)

stable_audio_open
musicldm
mustango
audioldm2
MusicGen_medium


The same file name appears in all generator folders. This means that the same MusicCaps example or prompt was used to generate audio with different models. This is useful for our project because we can compare how different generators produce music from the same input description. It also means that samples with the same file name should stay in the same train, validation, or test split to avoid data leakage.

In [13]:
import pandas as pd
import numpy as np

records = []

N_FILES = 100

for folder in generator_dirs:
    wav_files = list(folder.glob("*.wav"))[:N_FILES]

    for wav_file in wav_files:
        try:
            audio, sr = sf.read(wav_file)

            duration = len(audio) / sr
            channels = 1 if audio.ndim == 1 else audio.shape[1]
            max_amplitude = np.max(np.abs(audio))
            rms = np.sqrt(np.mean(audio ** 2))

            records.append({
                "generator": folder.name,
                "file": wav_file.name,
                "sample_rate": sr,
                "duration": duration,
                "channels": channels,
                "dtype": str(audio.dtype),
                "max_amplitude": max_amplitude,
                "rms": rms,
            })

        except Exception as e:
            print(f"Could not read {wav_file}: {e}")

df = pd.DataFrame(records)
df.head()

,generator,file,sample_rate,duration,channels,dtype,max_amplitude,rms
0,stable_audio_open,dHGAXJ9RPJs.wav,16000,10.0,1,float64,0.939575,0.184742
1,stable_audio_open,kjcXM60HrRA.wav,16000,10.0,1,float64,1.074754,0.328285
2,stable_audio_open,MpWGx5odhh8.wav,16000,10.0,1,float64,0.868102,0.137809
3,stable_audio_open,aItOEVYnpHA.wav,16000,10.0,1,float64,0.628708,0.116032
4,stable_audio_open,6elnFyuUnaU.wav,16000,10.0,1,float64,0.735204,0.143004


This analysis confirms that all sampled files use a 16 kHz sample rate and one audio channel. Duration is constant within each generator: 10.0 s for AudioLDM2, MusicLDM and Stable Audio Open, 10.18 s for MusicGen Medium, and 10.242 s for Mustango. 

In [14]:
df.groupby("generator")["duration"].agg(["min", "max", "mean", "std"])

,min,max,mean,std
generator,,,,
MusicGen_medium,10.180,10.180,10.180,0.0
audioldm2,10.000,10.000,10.000,0.0
musicldm,10.000,10.000,10.000,0.0
mustango,10.242,10.242,10.242,0.0
stable_audio_open,10.000,10.000,10.000,0.0


In [15]:
df.groupby("generator")["sample_rate"].unique()

generator
MusicGen_medium      [16000]
audioldm2            [16000]
musicldm             [16000]
mustango             [16000]
stable_audio_open    [16000]
Name: sample_rate, dtype: object

In [16]:
df.groupby("generator")["channels"].unique()

generator
MusicGen_medium      [1]
audioldm2            [1]
musicldm             [1]
mustango             [1]
stable_audio_open    [1]
Name: channels, dtype: object

In [17]:
df.groupby("generator")["max_amplitude"].agg(["min", "max", "mean"])

,min,max,mean
generator,,,
MusicGen_medium,0.049587,1.023177,0.554553
audioldm2,0.261719,0.841309,0.546697
musicldm,0.121887,0.978516,0.555197
mustango,0.375641,0.995239,0.678988
stable_audio_open,0.136415,1.775615,0.753946


In [18]:
df.groupby("generator")["rms"].agg(["min", "max", "mean"])

,min,max,mean
generator,,,
MusicGen_medium,0.008109,0.289291,0.099326
audioldm2,0.009058,0.150833,0.090340
musicldm,0.015697,0.199695,0.098360
mustango,0.035532,0.357337,0.115505
stable_audio_open,0.017792,0.380981,0.140748


In [19]:
df[df["max_amplitude"] > 1].groupby("generator").size()

generator
MusicGen_medium       2
stable_audio_open    21
dtype: int64

Some audio files have maximum absolute amplitude values above 1. This occurs mainly for stable_audio_open in the current sample. We will not normalize these files yet, because this may be a systematic property of the generated audio and could be relevant for further analysis.

In [20]:
df[df["max_amplitude"] > 1][
    ["generator", "file", "max_amplitude", "rms", "duration"]
].sort_values("max_amplitude", ascending=False)

,generator,file,max_amplitude,rms,duration
36,stable_audio_open,tPVsTFGWU5Q.wav,1.775615,0.380981,10.00
51,stable_audio_open,PRzBkZSSyY0.wav,1.494021,0.194129,10.00
82,stable_audio_open,L2-EGNKzUAQ.wav,1.466395,0.199123,10.00
31,stable_audio_open,nbb8vmfjHJs.wav,1.317819,0.219064,10.00
41,stable_audio_open,BxQSEvHdyjQ.wav,1.314933,0.170301,10.00
78,stable_audio_open,QrKJs6lBfmM.wav,1.270347,0.129617,10.00
69,stable_audio_open,I-xPuRe9vF0.wav,1.187683,0.312825,10.00
27,stable_audio_open,xe9vDoF8TQM.wav,1.168455,0.269974,10.00
65,stable_audio_open,_43OOP6UEw0.wav,1.152641,0.178689,10.00
16,stable_audio_open,2-K-7T8ZIWA.wav,1.132906,0.332884,10.00


### Full dataset technical analysis

In [21]:
records_full = []

for folder in generator_dirs:
    wav_files = list(folder.glob("*.wav"))

    for wav_file in wav_files:
        try:
            audio, sr = sf.read(wav_file)

            duration = len(audio) / sr
            channels = 1 if audio.ndim == 1 else audio.shape[1]
            max_amplitude = np.max(np.abs(audio))
            rms = np.sqrt(np.mean(audio ** 2))

            records_full.append({
                "generator": folder.name,
                "file": wav_file.name,
                "sample_rate": sr,
                "duration": duration,
                "channels": channels,
                "dtype": str(audio.dtype),
                "max_amplitude": max_amplitude,
                "rms": rms,
            })

        except Exception as e:
            print(f"Could not read {wav_file}: {e}")

df_full = pd.DataFrame(records_full)

df_full.shape

(27605, 8)

In [19]:
df_full.groupby("generator")["duration"].agg(["min", "max", "mean", "std"])

,min,max,mean,std
generator,,,,
MusicGen_medium,10.180,10.180,10.180,0.0
audioldm2,10.000,10.000,10.000,0.0
musicldm,10.000,10.000,10.000,0.0
mustango,10.242,10.242,10.242,0.0
stable_audio_open,10.000,10.000,10.000,0.0


In [20]:
df_full.groupby("generator")["sample_rate"].unique()

generator
MusicGen_medium      [16000]
audioldm2            [16000]
musicldm             [16000]
mustango             [16000]
stable_audio_open    [16000]
Name: sample_rate, dtype: object

In [21]:
df_full.groupby("generator")["channels"].unique()

generator
MusicGen_medium      [1]
audioldm2            [1]
musicldm             [1]
mustango             [1]
stable_audio_open    [1]
Name: channels, dtype: object

In [22]:
df_full.groupby("generator")["max_amplitude"].agg(["min", "max", "mean"])

,min,max,mean
generator,,,
MusicGen_medium,0.001550,1.778568,0.575105
audioldm2,0.022476,0.993652,0.547040
musicldm,0.002171,0.994141,0.547359
mustango,0.176666,1.000000,0.699738
stable_audio_open,0.014892,2.306482,0.744138


In [23]:
df_full.groupby("generator")["rms"].agg(["min", "max", "mean"])

,min,max,mean
generator,,,
MusicGen_medium,0.000097,0.485937,0.098470
audioldm2,0.002596,0.294633,0.094104
musicldm,0.000246,0.359610,0.098955
mustango,0.010974,0.752150,0.119573
stable_audio_open,0.001554,0.733426,0.140802


In [24]:
df_full[df_full["max_amplitude"] > 1].groupby("generator").size()

generator
MusicGen_medium       445
stable_audio_open    1127
dtype: int64

In [22]:
df_full.assign(over_1=df_full["max_amplitude"] > 1).groupby("generator")["over_1"].mean()  * 100

generator
MusicGen_medium       8.060134
audioldm2             0.000000
musicldm              0.000000
mustango              0.000000
stable_audio_open    20.412969
Name: over_1, dtype: float64

The full dataset analysis confirms the main results from the smaller sample. All 27,605 audio files use a sample rate of 16 kHz and have one audio channel. This means that we do not need to resample the audio or convert stereo files to mono.

The duration is the same inside each generator, but it is slightly different between generators. AudioLDM2, MusicLDM and Stable Audio Open have 10.0-second clips, MusicGen Medium has 10.18-second clips, and Mustango has 10.242-second clips. Because of this, duration itself could give the model a clue about the generator, so it will be handled carefully during preprocessing.

We can also see differences in amplitude and RMS values between generators. Stable Audio Open has the highest average amplitude and RMS, while Mustango also has slightly higher values than the other generators.
Some files also have maximum amplitude values above 1. This happens for 445 MusicGen Medium files, which is about 8.1% of the class, and for 1,127 Stable Audio Open files, which is about 20.4%. No such files were found for AudioLDM2, MusicLDM or Mustango.

These differences may come from the way the models generate or save the audio. Because of this, we should not apply amplitude normalization automatically. It is better to keep the original values for now and later test whether these differences help the model identify the generator.

Overall, the dataset is already consistent in sample rate and number of channels. The main preprocessing questions are how to handle the small differences in duration and whether amplitude normalization should be used. These choices will be tested carefully so that the model doesn't rely only on simple technical differences between generators.

### Preprocessing test

In [5]:
from ai_music_origin.data import load_audio, preprocess_audio
test_file = next(generator_dirs[0].glob("*.wav"))

audio, sr = load_audio(test_file)

print("Original shape:", audio.shape)
print("Sample rate:", sr)
print("Original duration:", len(audio) / sr)

Original shape: (160000,)
Sample rate: 16000
Original duration: 10.0


In [6]:
processed = preprocess_audio(
    audio,
    sr,
    target_duration=10.0,
    normalize=False,
)

print("Processed shape:", processed.shape)
print("Processed duration:", len(processed) / sr)

Processed shape: (160000,)
Processed duration: 10.0


In [7]:
normalized = preprocess_audio(
    audio,
    sr,
    normalize=True,
)

print("Original max amplitude:", np.max(np.abs(audio)))
print("Normalized max amplitude:", np.max(np.abs(normalized)))

Original max amplitude: 0.9395753
Normalized max amplitude: 1.0


In [8]:
mustango_dir = DATA_DIR / "mustango"
test_file_long = next(mustango_dir.glob("*.wav"))

audio_long, sr_long = load_audio(test_file_long)

processed_long = preprocess_audio(
    audio_long,
    sr_long,
    target_duration=10.0,
    normalize=False,
)

print("Original duration:", len(audio_long) / sr_long)
print("Processed duration:", len(processed_long) / sr_long)

Original duration: 10.242
Processed duration: 10.0


The preprocessing functions work correctly on the tested audio files. Audio can be loaded without changing the original data, and optional preprocessing steps such as trimming, padding, and amplitude normalization can be applied when needed. These steps are not applied by default.